In [ ]:
import pandas as pd
import os

In [ ]:
#dataset directory
data_dir = 'dataset/'
# File paths to get all the data
fake_real_fake_path = os.path.join(data_dir, 'Fake.csv')
fake_real_true_path = os.path.join(data_dir, 'True.csv')
welfake_path = os.path.join(data_dir, 'WELFake_Dataset.csv')

In [ ]:
#load the dataset
df_fake_real_fake = pd.read_csv(fake_real_fake_path)
df_fake_real_true = pd.read_csv(fake_real_true_path)

#create labels to differinate between true or false
df_fake_real_fake['label'] = 0 
df_fake_real_true['label'] = 1 

#combine the fake and true into one pd
df_fake_real = pd.concat([df_fake_real_fake, df_fake_real_true], ignore_index=True)

print(f"Loaded Fake and Real News Dataset: {df_fake_real.shape[0]} samples")

#load the other dataset no labels needed since its already part of the data
df_welfake = pd.read_csv(welfake_path)
print(f"Loaded WELFake Dataset: {df_welfake.shape[0]} samples")

In [ ]:
#print out the column names in the dataset
print("\nColumns in Fake and Real News Dataset:", df_fake_real.columns)
print("Columns in WELFake Dataset:", df_welfake.columns)

#find the total number of fake and real news in each dataset
print("\nCounts in Fake and Real News Dataset:\n", df_fake_real['label'].value_counts())
print("Counts in WELFake Dataset :\n", df_welfake['label'].value_counts())

In [ ]:
#combine dataset

#select the 3 main column needed
cols_to_use = ['title', 'text', 'label']
df_fake_real_subset = df_fake_real[cols_to_use]
df_welfake_subset = df_welfake[cols_to_use]

#combine the 2 into 1 pd
df_combined = pd.concat([df_fake_real_subset, df_welfake_subset], ignore_index=True)
print(f"Combined dataset size: {df_combined.shape[0]} samples")

In [ ]:

initial_samples = df_combined.shape[0]
df_combined.drop_duplicates(subset=['text'], keep='first', inplace=True)
samples_after_deduplication = df_combined.shape[0]

#samples removed in duplication
print(f"Samples removed after duplication: {initial_samples - samples_after_deduplication} samples")
#new sample size
print(f"New size after removing duplicates: {samples_after_deduplication} samples")

In [ ]:
#missing values check
print(f"Missing values:")
print(df_combined.isnull().sum())

#remove samples with missing text as it what we are mainly comparing
initial_samples_after_dedup = df_combined.shape[0]
df_combined.dropna(subset=['text'], inplace=True)
samples_after_dropna = df_combined.shape[0]

#samples removed during missing processing
if initial_samples_after_dedup - samples_after_dropna > 0:
    print(f"Samples removed: {initial_samples_after_dedup - samples_after_dropna}")
    print(f"New size after removing missing texts: {samples_after_dropna} samples")
else:
     print("No samples removed.")

In [ ]:
# Load the merged dataset to verify, if needed
# df = pd.read_csv("data/news_dataset.csv")

In [ ]:
#data summary
print("\nFinal Combined Dataset Summary")
print(f"Total samples: {df_combined.shape[0]}")
#distrubtion
print("\nTrue and Fake Distrubtion:")
print(df_combined['label'].value_counts())
#sample summary
print("\nSample summary:")
print(df_combined.head())

#new index after dropping rows
df_combined.reset_index(drop=True, inplace=True)

In [ ]:
import spacy
import re

#pretrained nlp model with english word
nlp = spacy.load("en_core_web_sm")
#lets us load our words
custom_stop_words = set()

#cleaning text input
def preprocess_text(text):
    #empty for non-string inputs
    if not isinstance(text, str):
        return ""

    #lowercase everything and remove whitespaces
    text = text.lower().strip()

    #remove any urls 
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    #remove any character that is not a lowercase or spae
    text = re.sub(r'[^a-z\s]', '', text)

    #use spacy to tokenize the text
    doc = nlp(text) 

    cleaned_tokens = []
    for token in doc:
        #ignore punctation, whitespace, stopwords, and non-alpha
        if not token.is_punct and not token.is_space and not token.is_stop and token.is_alpha:
             #append lemmatized form of token
             cleaned_tokens.append(token.lemma_)
    return cleaned_tokens 
    
#count number of sentences
def count_sentences(text):
    #return 0 if input is non-empty string
    if not isinstance(text, str) or not text.strip():
        return 0
    #return 0 if nlp pipeline is not loading
    if nlp_sent is None: 
        return 0
    #process text and count sentences
    doc = nlp_sent(text)
    return len(list(doc.sents))

#calculate avg tokens per sentence
def avg_sentence_length_processed(processed_tokens, sentence_count):
    #edgecase against division by zero or missing tokens
    if sentence_count is None or sentence_count == 0 or not processed_tokens:
        return 0.0
    total_tokens = len(processed_tokens)
    return total_tokens / sentence_count

#calculate proportion of uppercase
def uppercase_char_proportion(text):
    #invalid or empty input
    if not isinstance(text, str) or not text.strip():
         return 0.0
    #find all alpha characters
    alpha_chars = re.findall(r'[a-zA-Z]', text)
    if not alpha_chars: return 0.0
    #count uppercase and divide by total alpha characters
    upper_count = sum(1 for char in alpha_chars if char.isupper())
    return upper_count / len(alpha_chars)

#calculate the number of a certian punctation in text
def count_punctuation(text, punct_char='!'):
    #invalid empty input
    if not isinstance(text, str):
        return 0
    return text.count(punct_char)

from tqdm.auto import tqdm
from pandarallel import pandarallel 

#calculate number of cores and set workers for pandarallel
num_cores = os.cpu_count()
workers_to_use = max(1, num_cores - 2) #at least 1 core
pandarallel.initialize(progress_bar=True, nb_workers=workers_to_use)
print(f"Initializing pandarallel with {workers_to_use} workers")

#apply the preprocessing function to text and title columns parallely
df_combined['processed_text'] = df_combined['text'].parallel_apply(preprocess_text)
df_combined['processed_title'] = df_combined['title'].parallel_apply(preprocess_text)


#count sentence in each text
df_combined['sentence_count'] = df_combined['text'].parallel_apply(count_sentences)
   
#calculate average sentence length
df_combined['avg_sentence_length'] = df_combined.apply(
             lambda row: avg_sentence_length_processed(row.get('processed_text', []), row.get('sentence_count', 0)),
             axis=1
        )

#uppercase characters
df_combined['uppercase_char_prop'] = df_combined['text'].parallel_apply(uppercase_char_proportion)
#exclamation count
df_combined['exclamation_count'] = df_combined['text'].parallel_apply(lambda x: count_punctuation(x, '!'))
#question count
df_combined['question_count'] = df_combined['text'].parallel_apply(lambda x: count_punctuation(x, '?'))
#quotes count
df_combined['quotes_count'] = df_combined['text'].parallel_apply(lambda x: count_punctuation(x, '"') + count_punctuation(x, "'"))

#sample of the processed data
print("\nProcessed data with new features:")
print(df_combined.head())

#path for saving data
data_dir = 'dataset/'
processed_data_path = os.path.join(data_dir, 'combined_processed_articles.pkl')

#save all the data
print(f"\nSaving processed data to {processed_data_path}...")
df_combined.to_pickle(processed_data_path)
print("Succesfully saved data")

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from wordcloud import WordCloud
import nltk
from nltk import ngrams

#data path
data_dir = 'dataset/'
processed_data_path = os.path.join(data_dir, 'combined_processed_articles.pkl')

#load the data
print(f"Loading processed data from {processed_data_path}")
df_combined = pd.read_pickle(processed_data_path)
print("Succesfully loaded data!")
print(f"Shape: {df_combined.shape}")
print("Columns:", df_combined.columns)

print("\nEDA:")

#check all req columns
required_cols = [
    'label', 'processed_text', 'processed_title',
    'sentence_count', 'avg_sentence_length',
    'uppercase_char_prop', 'exclamation_count',
    'question_count', 'quotes_count',
    'processed_text_length'
]
#make sure all column names are present
for col in required_cols:
    if col not in df_combined.columns:
        print(f"⚠️ Missing column: {col}")

#make one single token list with all processed text
all_tokens = [
    token
    for doc in df_combined['processed_text'].dropna()
    for token in doc
]
token_counts = Counter(all_tokens)

#seperate tokens by label fake and real news
fake_tokens = [
    token
    for doc in df_combined[df_combined['label'] == 0]['processed_text'].dropna()
    for token in doc
]
real_tokens = [
    token
    for doc in df_combined[df_combined['label'] == 1]['processed_text'].dropna()
    for token in doc
]
fake_token_counts = Counter(fake_tokens)
real_token_counts = Counter(real_tokens)

#generate n-grams from given token list
def get_ngrams(tokens, n):
    return list(ngrams(tokens, n)) if len(tokens) >= n else []

#build bigrams for fake and real news
fake_bigrams = [
    bg
    for doc in df_combined[df_combined['label']==0]['processed_text'].dropna()
    for bg in get_ngrams(doc, 2)
]
real_bigrams = [
    bg
    for doc in df_combined[df_combined['label']==1]['processed_text'].dropna()
    for bg in get_ngrams(doc, 2)
]

#put tokens back into strings for word cloud
fake_str = " ".join(all_tokens for all_tokens in fake_tokens)
real_str = " ".join(all_tokens for all_tokens in real_tokens)

plt.figure(figsize=(16,6))
#create fake news cloud
plt.subplot(1,2,1)
WordCloud(width=800, height=400, background_color='white', max_words=100).generate(fake_str)
plt.imshow(WordCloud().generate(fake_str), interpolation='bilinear')
plt.axis('off'); plt.title('Fake News Word Cloud')

#create real news cloud
plt.subplot(1,2,2)
plt.imshow(WordCloud().generate(real_str), interpolation='bilinear')
plt.axis('off'); plt.title('Real News Word Cloud')
plt.tight_layout()
plt.show()


fig, axes = plt.subplots(1, 3, figsize=(18, 5))

#text length distrubtion fake or real bar graph
sns.histplot(
    df_combined, x='processed_text_length', hue='label',
    kde=True, multiple="layer", alpha=0.5,
    bins=30, ax=axes[0]
)
axes[0].set(
    title='Text Length',
    xlabel='Tokens', ylabel='Count'
)
axes[0].legend(title='Label', labels=['Fake','Real'])

#sentence count distrubtion fake or real bar graph
sns.histplot(
    df_combined, x='sentence_count', hue='label',
    discrete=False, multiple="dodge", shrink=0.8,
    bins=20, ax=axes[1]
)
axes[1].set(
    title='Sentence Count',
    xlabel='# Sentences', ylabel='Count'
)
axes[1].legend(title='Label', labels=['Fake','Real'])

#average sentence length distrubtion fake or real bar graph
sns.histplot(
    df_combined, x='avg_sentence_length', hue='label',
    kde=False, multiple="layer", alpha=0.5,
    bins=30, ax=axes[2]
)
axes[2].set(
    title='Avg Sentence Length',
    xlabel='Tokens per Sentence', ylabel='Count'
)
axes[2].legend(title='Label', labels=['Fake','Real'])

plt.tight_layout()
plt.show()


fig, axes = plt.subplots(2, 2, figsize=(16, 10))
axes = axes.flatten()

#uppercase character distrubtion fake or real bar graph
sns.histplot(
    df_combined, x='uppercase_char_prop', hue='label',
    multiple="layer", alpha=0.5, bins=30, ax=axes[0]
)
axes[0].set(
    title='Uppercase Char Proportion',
    xlabel='Proportion', ylabel='Count'
)
axes[0].legend(title='Label', labels=['Fake','Real'])

#exclamation mark distrubtion fake or real bar graph
ex = df_combined[df_combined['exclamation_count'] > 0]
sns.histplot(
    ex, x='exclamation_count', hue='label',
    discrete=True, multiple="dodge", shrink=0.8,
    ax=axes[1]
)
axes[1].set(
    title='Exclamation Marks',
    xlabel='# Exclamations', ylabel='Count'
)
axes[1].legend(title='Label', labels=['Fake','Real'])

#question mark distrubtion fake or real bar graph
q = df_combined[df_combined['question_count'] > 0]
sns.histplot(
    q, x='question_count', hue='label',
    discrete=True, multiple="dodge", shrink=0.8,
    ax=axes[2]
)
axes[2].set(
    title='Question Marks',
    xlabel='# Questions', ylabel='Count'
)
axes[2].legend(title='Label', labels=['Fake','Real'])

#quotes distrubtion fake or real bar graph
qt = df_combined[df_combined['quotes_count'] > 0]
sns.histplot(
    qt, x='quotes_count', hue='label',
    discrete=True, multiple="dodge", shrink=0.8,
    ax=axes[3]
)
axes[3].set(
    title='Quotes Count (>0)',
    xlabel='# Quotes', ylabel='Count'
)
axes[3].legend(title='Label', labels=['Fake','Real'])


plt.tight_layout()
plt.show()


#find top n tokens for fake and real
n_top = 20
top_fake = pd.DataFrame(fake_token_counts.most_common(n_top), columns=['token','count'])
top_real = pd.DataFrame(real_token_counts.most_common(n_top), columns=['token','count'])

# convert bigrams to strings
fb = pd.DataFrame(Counter(fake_bigrams).most_common(n_top), columns=['bigram','count'])
fb['bigram'] = fb['bigram'].apply(lambda x: ' '.join(x))
rb = pd.DataFrame(Counter(real_bigrams).most_common(n_top), columns=['bigram','count'])
rb['bigram'] = rb['bigram'].apply(lambda x: ' '.join(x))

fig, axes = plt.subplots(2, 2, figsize=(18, 12))
ax1, ax2, ax3, ax4 = axes.flatten()

#plot top fake token
sns.barplot(x='count', y='token', data=top_fake, ax=ax1, color='skyblue')
ax1.set(title='Top Fake Tokens', xlabel='Count', ylabel='Token')

#plot top real tokens
sns.barplot(x='count', y='token', data=top_real, ax=ax2, color='lightcoral')
ax2.set(title='Top Real Tokens', xlabel='Count', ylabel='Token')

#plot top fake bigrams
sns.barplot(x='count', y='bigram', data=fb, ax=ax3, color='skyblue')
ax3.set(title='Top Fake Bigrams', xlabel='Count', ylabel='Bigram')

#plot top real bigrams
sns.barplot(x='count', y='bigram', data=rb, ax=ax4, color='lightcoral')
ax4.set(title='Top Real Bigrams', xlabel='Count', ylabel='Bigram')

plt.tight_layout()
plt.show()

print("EDA Done!")